In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping



warnings.simplefilter(action="ignore", category=FutureWarning)

In [13]:
import os
import random

def fijar_semillas(semilla=42):
    # 1. Fijar semilla de Python
    os.environ['PYTHONHASHSEED'] = str(semilla)
    random.seed(semilla)
    
    # 2. Fijar semilla de NumPy
    np.random.seed(semilla)
    
    # 3. Fijar semilla de TensorFlow/Keras
    tf.random.set_seed(semilla)
    
    print(f"[*] Semillas fijadas a {semilla} para asegurar reproducibilidad.")

# Llamar a la función antes de crear ningún modelo ni dividir datos
fijar_semillas(42)

[*] Semillas fijadas a 42 para asegurar reproducibilidad.


In [14]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS (Código del Profesor)
# =====================================================================
print("Descargando datos de Yahoo Finance...")
start_date = '1960-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

# Cálculo de retornos logarítmicos
returns = np.log(precios_close).diff().dropna()
print(f"Forma de los datos de retornos: {returns.shape}")

# Función del profesor para crear ventanas
def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    num_features = data_array.shape[1] 

    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_sequence = data_array[i : i + input_window_size]
        X.append(input_sequence)
        
        if output_window_size > 0:
            output_sequence = data_array[i + input_window_size : i + input_window_size + output_window_size]
            average_output = np.mean(output_sequence, axis=0) 
            y.append(average_output)
        else:
            y.append(data_array[i + input_window_size - 1])
            
    return np.array(X), np.array(y)

Descargando datos de Yahoo Finance...


c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

Forma de los datos de retornos: (16190, 23)


In [15]:
# =====================================================================
# 2. DEFINICIÓN DE ARQUITECTURA Y BASELINES
# =====================================================================

def construir_modelo_rnn(config, input_shape, n_assets=23):
    """Construye un modelo dinámico basado en la configuración dada."""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # Capa dinámica (LSTM o GRU)
    CapaRecurrente = config['tipo_capa']
    model.add(CapaRecurrente(config['neuronas'], return_sequences=False))
    
    model.add(Dropout(config['dropout']))
    model.add(Dense(n_assets)) # Salida: 23 valores (promedio de los 23 activos)
    
    optimizador = Adam(learning_rate=config['lr'])
    model.compile(optimizer=optimizador, loss='mae')
    return model

def calcular_baselines(X_test, y_test, y_train_mean):
    """Calcula el MAE para modelos simples, incluyendo Buy and Hold."""
    
    # 1. Baseline Naive: El futuro será igual al último día de la ventana de entrada
    y_pred_naive = X_test[:, -1, :]
    mae_naive = np.mean(np.abs(y_pred_naive - y_test))
    
    # 2. Baseline SMA: El futuro será igual a la media de la ventana de entrada actual
    y_pred_sma = np.mean(X_test, axis=1)
    mae_sma = np.mean(np.abs(y_pred_sma - y_test))
    
    # 3. Baseline Buy and Hold: Predecir siempre la media histórica del entrenamiento
    # Creamos un array del mismo tamaño que y_test relleno con la media de y_train
    y_pred_bh = np.full_like(y_test, y_train_mean)
    mae_bh = np.mean(np.abs(y_pred_bh - y_test))
    
    return mae_naive, mae_sma, mae_bh

# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia', exist_ok=True)

In [19]:
# =====================================================================
# 3. CONFIGURACIÓN DEL EXPERIMENTO
# =====================================================================

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

# 1. Lista para ventanas con POCA información (In: 5 y 10)
# Definimos el punto de partida común para clonarlo fácilmente
base_configs = [
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.0, 'lr': 0.001}
]

# Inicializamos las 8 listas independientes. 
# Usamos list() para que sean copias independientes y puedas modificarlas en el futuro
hp_in5_corto  = [
    # Ganador IN 5 OUT 1  |||| IN 5 OUT 5  
    {'tipo_capa': GRU,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.001},

    {'tipo_capa': GRU,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.0001},

    {'tipo_capa': GRU,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.00001},

    {'tipo_capa': GRU,  'neuronas': 6, 'dropout': 0.0, 'lr': 0.0001},

]

hp_in5_largo  = [
    # Ganador IN 5 OUT 30 
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.1, 'lr': 0.001},

    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.1, 'lr': 0.0005},

    # Ganador IN 5 OUT 90 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.3, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.25, 'lr': 0.000005}
]

hp_in10_corto = [
    
    # Ganador IN 10 OUT 1 ||||| IN 10 OUT 5
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.1, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.1, 'lr': 0.00005},

    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.2, 'lr': 0.0005},
]


hp_in10_largo = [
  
    # Ganador IN 10 OUT 30 
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.2, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.2, 'lr': 0.00005},

    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.3, 'lr': 0.0005},
    
    # Ganador IN 10 OUT 90 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.2, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.25, 'lr': 0.00005},

    {'tipo_capa': LSTM, 'neuronas': 256, 'dropout': 0.2, 'lr': 0.0005}

]

hp_in30_corto = [
    # Ganador IN 30 OUT 1 
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.2, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.3, 'lr': 0.00005},
    
    # Ganador IN 30 OUT 5 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.2, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.3, 'lr': 0.00005},

    {'tipo_capa': LSTM, 'neuronas': 256, 'dropout': 0.2, 'lr': 0.0005}
]


hp_in30_largo = [
    # Ganador IN 30 OUT 30  |||| Ganador IN 30 OUT 90 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.2, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.3, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 256, 'dropout': 0.2, 'lr': 0.0005},
    
]

hp_in90_corto = [
    # Ganador IN 90 OUT 1 ||||| IN 90 OUT 5
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.001},

    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.1, 'lr': 0.001},

    {'tipo_capa': LSTM, 'neuronas': 32, 'dropout': 0.1, 'lr': 0.001},

    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.1, 'lr': 0.001},

]

hp_in90_largo = [
    # Ganador IN 90 OUT 30 ||||| IN 90 OUT 90
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.1, 'lr': 0.0005},

    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.2, 'lr': 0.00005},

    {'tipo_capa': LSTM, 'neuronas': 256, 'dropout': 0.1, 'lr': 0.0005}
    
]

lista_hiperparametros = []

# Matrices para reportar resultados finales de las Redes Recurrentes
matriz_mae_train_rnn = np.zeros((4, 4))
matriz_mae_val_rnn = np.zeros((4, 4))
matriz_mae_rnn = np.zeros((4, 4)) # Esta es la de Test que ya tenías

matriz_mae_naive = np.zeros((4, 4))
matriz_mae_sma = np.zeros((4, 4))
matriz_mae_bh = np.zeros((4, 4))

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [22]:
# =====================================================================
# 4. BUCLE PRINCIPAL (AUTOMATIZACIÓN DE LOS 16 MODELOS x CONFIGURACIONES)
# =====================================================================


# =====================================================================
# RECORDATORIO: Inicializa estas nuevas matrices antes del bucle
# =====================================================================
matriz_mae_naive_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_sma_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_bh_val = np.zeros((len(input_windows), len(output_windows)))

print("\nIniciando entrenamiento de modelos...")

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):
        print(f"\n=======================================================")
        print(f" Ventana Entrada: {in_w} días | Ventana Salida: {out_w} días")
        print(f"=======================================================")
        
        # 1. Crear datos
        X, y = create_time_series_data(returns, in_w, out_w)
        
        # 2. Separación CRONOLÓGICA: 80% Train, 10% Validacion, 10% Test
        # split_1 = int(len(X) * 0.8)
        # split_2 = int(len(X) * 0.9)
        
        # Para un esquema 70% Train, 20% Validacion, 10% Test
        split_1 = int(len(X) * 0.70) # Aquí cortamos el Train
        split_2 = int(len(X) * 0.90) # Aquí cortamos la Validación (del 70% al 90% = 20%)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]

        # Calculamos la media global de entrenamiento para esta ventana (Buy and Hold)
        y_train_mean = np.mean(y_train, axis=0)
        
        
        # =====================================================================
        # 3. Baselines (AHORA EN VALIDACIÓN Y TEST)
        # =====================================================================

        '''
        mae_naive, mae_sma, mae_bh = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive
        matriz_mae_sma[i, j] = mae_sma
        matriz_mae_bh[i, j] = mae_bh

        print(f"Baseline Naive (MAE en Test): {mae_naive:.6f}")
        print(f"Baseline SMA   (MAE en Test): {mae_sma:.6f}")
        print(f"Baseline Buy & Hold (MAE): {mae_bh:.6f}")
        '''

        # Calcular en Validación
        mae_naive_val, mae_sma_val, mae_bh_val = calcular_baselines(X_val, y_val, y_train_mean)
        matriz_mae_naive_val[i, j] = mae_naive_val
        matriz_mae_sma_val[i, j] = mae_sma_val
        matriz_mae_bh_val[i, j] = mae_bh_val
        
        # Calcular en Test
        mae_naive_test, mae_sma_test, mae_bh_test = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive_test
        matriz_mae_sma[i, j] = mae_sma_test
        matriz_mae_bh[i, j] = mae_bh_test
        

        print("--- Baselines VALIDACIÓN ---")
        print(f"Naive: {mae_naive_val:.6f} | SMA: {mae_sma_val:.6f} | Buy&Hold: {mae_bh_val:.6f}")
        print("--- Baselines TEST ---")
        print(f"Naive: {mae_naive_test:.6f} | SMA: {mae_sma_test:.6f} | Buy&Hold: {mae_bh_test:.6f}\n")


        # 4. Búsqueda del mejor modelo recurrente
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None


        # DIVIDE Y VENCERÁS: Selección de hiperparámetros
        if in_w == 5:
            lista_a_probar = hp_in5_corto if out_w in [1, 5] else hp_in5_largo
            nombre_lista = "In:5 Corto" if out_w in [1, 5] else "In:5 Largo"
            
        elif in_w == 10:
            lista_a_probar = hp_in10_corto if out_w in [1, 5] else hp_in10_largo
            nombre_lista = "In:10 Corto" if out_w in [1, 5] else "In:10 Largo"
            
        elif in_w == 30:
            lista_a_probar = hp_in30_corto if out_w in [1, 5] else hp_in30_largo
            nombre_lista = "In:30 Corto" if out_w in [1, 5] else "In:30 Largo"
            
        elif in_w == 90:
            lista_a_probar = hp_in90_corto if out_w in [1, 5] else hp_in90_largo
            nombre_lista = "In:90 Corto" if out_w in [1, 5] else "In:90 Largo"

        print(f" -> Usando banco de pruebas: [{nombre_lista}]")
        
        for config in lista_a_probar:
            capa_nombre = config['tipo_capa'].__name__
            print(f" -> Entrenando: {capa_nombre}, Neuronas: {config['neuronas']}, LR: {config['lr']}, DropOut: {config['dropout']}")
            
            modelo = construir_modelo_rnn(config, input_shape=(in_w, 23))
            
            # Usamos verbose=0 para no llenar la pantalla de números, epochs=50 es suficiente con EarlyStop
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   # callbacks=[early_stop], 
                                   verbose=0
                                   )
            
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n[GANADOR] {mejor_config['tipo_capa'].__name__} ({mejor_config['neuronas']} neuronas)")
        
        # 5. Evaluación final del GANADOR en TRAIN, VALIDACIÓN y TEST
        mae_train_ganador = mejor_modelo.evaluate(X_train, y_train, verbose=0)
        mae_val_ganador = mejor_modelo.evaluate(X_val, y_val, verbose=0)
        mae_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        
        # Guardar en sus respectivas matrices
        matriz_mae_train_rnn[i, j] = mae_train_ganador
        matriz_mae_val_rnn[i, j] = mae_val_ganador
        matriz_mae_rnn[i, j] = mae_test_ganador
        
        print(f"MAE del Modelo Ganador en TRAIN:      {mae_train_ganador:.6f}")
        print(f"MAE del Modelo Ganador en VALIDACIÓN: {mae_val_ganador:.6f}")
        print(f"MAE del Modelo Ganador en TEST:       {mae_test_ganador:.6f}")
        
        # 6. Guardar Gráfica de Convergencia del Ganador
        plt.figure(figsize=(10, 5))
        plt.plot(mejor_historial.history['loss'], label='Error Entrenamiento (MAE)')
        plt.plot(mejor_historial.history['val_loss'], label='Error Validación (MAE)')
        
        # Título con todos los hiperparámetros
        nombre_capa = mejor_config['tipo_capa'].__name__
        n_neuronas = mejor_config['neuronas']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        
        plt.title(f"Convergencia {nombre_capa} | Neuronas: {n_neuronas} | LR: {l_rate} | Drop: {d_out}\n(Ventana In:{in_w} - Out:{out_w})")
        
        plt.xlabel('Épocas')
        plt.ylabel('MAE')
        plt.legend()
        plt.grid(True)
        
        # Guardar la imagen
        nombre_archivo = f"graficas_convergencia/conver_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos...

 Ventana Entrada: 5 días | Ventana Salida: 1 días
--- Baselines VALIDACIÓN ---
Naive: 0.015405 | SMA: 0.011786 | Buy&Hold: 0.010571
--- Baselines TEST ---
Naive: 0.017789 | SMA: 0.013608 | Buy&Hold: 0.012243

 -> Usando banco de pruebas: [In:5 Corto]
 -> Entrenando: GRU, Neuronas: 8, LR: 0.001, DropOut: 0.0
 -> Entrenando: GRU, Neuronas: 8, LR: 0.0001, DropOut: 0.0
 -> Entrenando: GRU, Neuronas: 8, LR: 1e-05, DropOut: 0.0
 -> Entrenando: GRU, Neuronas: 6, LR: 0.0001, DropOut: 0.0

[GANADOR] GRU (8 neuronas)
MAE del Modelo Ganador en TRAIN:      0.011835
MAE del Modelo Ganador en VALIDACIÓN: 0.010591
MAE del Modelo Ganador en TEST:       0.012268

 Ventana Entrada: 5 días | Ventana Salida: 5 días
--- Baselines VALIDACIÓN ---
Naive: 0.011802 | SMA: 0.006919 | Buy&Hold: 0.004730
--- Baselines TEST ---
Naive: 0.013656 | SMA: 0.008029 | Buy&Hold: 0.005580

 -> Usando banco de pruebas: [In:5 Corto]
 -> Entrenando: GRU, Neuronas: 8, LR: 0.001, DropOut: 

KeyboardInterrupt: 

In [21]:
# =====================================================================
# 5. RESULTADOS FINALES (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (RNN)")
print("="*50)
df_rnn_train = pd.DataFrame(matriz_mae_train_rnn, 
                            index=[f'In_{w}' for w in input_windows], 
                            columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_train)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (RNN)")
print("="*50)
df_rnn_val = pd.DataFrame(matriz_mae_val_rnn, 
                          index=[f'In_{w}' for w in input_windows], 
                          columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN TEST (RNN)")
print("="*50)
df_rnn = pd.DataFrame(matriz_mae_rnn, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_rnn)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (TEST)")
print("="*50)
df_naive = pd.DataFrame(matriz_mae_naive, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (VALIDACION)")
print("="*50)
df_naive_val = pd.DataFrame(matriz_mae_naive_val, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (TEST)")
print("="*50)
df_sma = pd.DataFrame(matriz_mae_sma, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (VALIDACION)")
print("="*50)
df_sma_val = pd.DataFrame(matriz_mae_sma_val, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (TEST)")
print("="*50)
df_bh = pd.DataFrame(matriz_mae_bh, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (TEST)")
print("="*50)
df_bh_val = pd.DataFrame(matriz_mae_bh_val, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh_val)



MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.011882  0.005526  0.002211  0.001286
In_10  0.012015  0.005757  0.002288  0.001341
In_30  0.011938  0.005551  0.002229  0.001283
In_90  0.011950  0.005616  0.002259  0.001337

MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.010628  0.004770  0.001949  0.001126
In_10  0.010700  0.004955  0.002024  0.001194
In_30  0.010656  0.004806  0.001985  0.001137
In_90  0.010671  0.004842  0.002007  0.001206

MATRIZ DE RESULTADOS FINALES EN TEST (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.012293  0.005606  0.002337  0.001285
In_10  0.012355  0.005793  0.002424  0.001379
In_30  0.012342  0.005675  0.002377  0.001302
In_90  0.012374  0.005736  0.002414  0.001380

MATRIZ DE RESULTADOS BASELINE NAIVE (TEST)
          Out_1     Out_5    Out_30    Out_90
In_5   0.017789  0.013656  0.012518  0.012254
In_10  0.017790  0.013657 